In [1]:
import requests
from tqdm import tqdm
import time
from datetime import datetime
import os
from dotenv import load_dotenv
import mwparserfromhell
import re
import json

# Data Fetching for TV Show plots

### Use TMDB API to fetch tv show released after the training date

In [2]:
# Fetch popular TV shows from TMDB (2024 to now)
# Load environment variables from .env file
load_dotenv()

# Get TMDB API key from environment variable
TMDB_API_KEY = os.getenv("TMDB_API_KEY")
if not TMDB_API_KEY:
    raise ValueError("TMDB_API_KEY not found in environment variables. Please create a .env file with your API key.")

BASE_URL = "https://api.themoviedb.org/3"

# Date range: 2024 to now (2025)
START_DATE = "2024-01-01"
END_DATE = datetime.now().strftime("%Y-%m-%d")  # Current date

# Fetch pages 51-150 (100 pages × 20 results per page = 2000 TV shows)
START_PAGE = 51
END_PAGE = 150

def fetch_tmdb_tv_shows(start_date, end_date, start_page=1, end_page=5, sort_by="popularity.desc"):
    """
    Fetch popular TV shows from TMDB within a date range and page range
    Returns list of TV show dictionaries with title, year, media_type, tmdb_id, and first_air_date
    """
    items = []
    total_pages = end_page - start_page + 1

    print(f"Fetching pages {start_page} to {end_page} from TMDB...")
    print(f"Date range: {start_date} to {end_date}")
    print(f"Sort: {sort_by}")
    print(f"Total pages: {total_pages} ({total_pages * 20} TV shows)\n")

    for page in tqdm(range(start_page, end_page + 1), desc="Fetching pages"):
        url = f"{BASE_URL}/discover/tv"
        params = {
            "api_key": TMDB_API_KEY,
            "language": "en-US",
            "sort_by": sort_by,
            "page": page,
            "first_air_date.gte": start_date,
            "first_air_date.lte": end_date
        }

        try:
            r = requests.get(url, params=params)
            r.raise_for_status()
            data = r.json()

            # Check if we've reached the end
            if not data.get("results"):
                print(f"\nNo more results at page {page}")
                break

            for tv_show in data["results"]:
                items.append({
                    "title": tv_show.get("name", ""),
                    "year": (tv_show.get("first_air_date") or "")[:4],
                    "first_air_date": tv_show.get("first_air_date", ""),
                    "media_type": "tv",
                    "tmdb_id": tv_show["id"]
                })

            # Small delay to be respectful to API
            time.sleep(0.25)

        except requests.exceptions.HTTPError as e:
            if e.response.status_code == 429:
                print(f"\n⚠️  Rate limited at page {page}. Waiting 10 seconds...")
                time.sleep(10)
                continue
            else:
                print(f"\n⚠️  Error at page {page}: {e}")
                break

    return items

# Fetch popular TV shows from 2024 to now (pages 51-150)
tv_shows = fetch_tmdb_tv_shows(START_DATE, END_DATE, start_page=START_PAGE, end_page=END_PAGE, sort_by="popularity.desc")
print(f"\n✓ Fetched {len(tv_shows)} TV shows from pages {START_PAGE}-{END_PAGE} (2024-{datetime.now().strftime('%Y')})")

Fetching pages 51 to 150 from TMDB...
Date range: 2024-01-01 to 2026-03-17
Sort: popularity.desc
Total pages: 100 (2000 TV shows)



Fetching pages: 100%|██████████| 100/100 [00:41<00:00,  2.42it/s]


✓ Fetched 2000 TV shows from pages 51-150 (2024-2026)


### Use Wikipedia API to fetch episode plot

In [3]:
# Wikipedia API Functions for TV Shows
WIKI_API = "https://en.wikipedia.org/w/api.php"

# Wikipedia requires a User-Agent header to identify your application
HEADERS = {
    "User-Agent": "WikipediaTVShowExtractor/1.0 (https://example.com/contact; your-email@example.com)"
}

def search_wikipedia(title, year=None, media_type=None):
    """Search for a Wikipedia page by title"""
    query = title
    if media_type == "tv":
        query += " TV series"
    elif media_type == "movie":
        query += " film"

    params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "format": "json"
    }

    try:
        r = requests.get(WIKI_API, params=params, headers=HEADERS)
        r.raise_for_status()
        results = r.json()["query"]["search"]
        time.sleep(0.5)  # Be respectful to Wikipedia's servers
        return results[0]["title"] if results else None
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 403:
            print(f"⚠️  403 Forbidden - Wikipedia may be blocking requests. Try again later.")
        raise

def get_wikipedia_episodes(page_title, debug=False):
    """
    Extract episode information from Wikipedia TV show page.
    Looks for episode tables, episode lists, and season sections.
    Returns a list of episode dictionaries with section name and content.
    """
    params = {
        "action": "query",
        "prop": "revisions",
        "rvprop": "content",
        "rvslots": "main",
        "titles": page_title,
        "format": "json"
    }

    try:
        r = requests.get(WIKI_API, params=params, headers=HEADERS)
        r.raise_for_status()
        pages = r.json()["query"]["pages"]
        time.sleep(0.5)  # Be respectful to Wikipedia's servers
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 403:
            if debug:
                print(f"⚠️  403 Forbidden - Wikipedia may be blocking requests.")
        raise

    page = next(iter(pages.values()))
    if "revisions" not in page:
        if debug:
            print(f"✗ No revisions found for page: {page_title}")
        return None

    text = page["revisions"][0]["slots"]["main"]["*"]
    wikicode = mwparserfromhell.parse(text)
    
    episodes = []
    
    # Get all sections (level 2 and level 3 headings for seasons)
    all_sections_level2 = wikicode.get_sections(levels=[2])
    all_sections_level3 = wikicode.get_sections(levels=[3])
    
    # First, manually extract section headings to find "Episodes"
    section_headings1 = {}
    for section in all_sections_level2:
        section_text = str(section)
        match = re.search(r'^==+\s*(.+?)\s*==+', section_text, re.MULTILINE)
        if match:
            heading = match.group(1).strip()
            section_headings1[heading] = section

    section_headings2 = {}
    for section in all_sections_level3:
        section_text = str(section)
        match = re.search(r'^==+\s*(.+?)\s*==+', section_text, re.MULTILINE)
        if match:
            heading = match.group(1).strip()
            section_headings2[heading] = section
    
    if debug:
        print(f"\nAvailable sections on '{page_title}':")
        for heading in list(section_headings1.keys())[:20]:
            print(f"  - {heading}")
    
    # Find "Episode" or "Episodes" section (case-insensitive)
    episodes_section = None
    episode_section_key = None
    
    # Check level 2 sections first
    for key in section_headings1.keys():
        if key.lower() in ["episodes", "episode", "episode list", "list of episodes"]:
            episodes_section = section_headings1[key]
            episode_section_key = key
            break
    
    # If not found, check level 3 sections
    if not episodes_section:
        for key in section_headings2.keys():
            if key.lower() in ["episodes", "episode", "episode list", "list of episodes"]:
                episodes_section = section_headings2[key]
                episode_section_key = key
                break
    
    if not episodes_section:
        if debug:
            print(f"✗ No 'Episode' or 'Episodes' section found")
        return None
    
    if debug:
        print(f"✓ Found '{episode_section_key}' section")
    
    # Get raw text for the section
    episodes_raw_text = str(episodes_section)
    
    # Extract cleaned text content from the section
    episodes_text_cleaned = episodes_section.strip_code().strip()
    
    if debug:
        print(f"  Raw text length: {len(episodes_raw_text)} characters")
        print(f"  Cleaned content length: {len(episodes_text_cleaned)} characters")
    
    # Return dictionary with section, content, and raw_wikicode
    return {
        "section": episode_section_key,
        "content": episodes_text_cleaned[:50000],  # Increased limit for unparsed content
        "raw_wikicode": episodes_raw_text[:50000]  # Also include raw wikicode for later parsing
    }

print("✓ Wikipedia functions loaded")

✓ Wikipedia functions loaded


In [ ]:
# Main extraction loop: Get episodes for all TV shows
OUTPUT_FILE = "../raw_data/tv_raw/tmdb_wikipedia_tv_episodes.json"
CHECKPOINT_FILE = "../raw_data/tv_raw/tv_episode_extraction_checkpoint.json"

# ensures raw_data/ and movie_raw/ both exist
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)


REQUEST_DELAY = 1.5  # Delay between requests (seconds)
BATCH_SAVE_INTERVAL = 5  # Save checkpoint every N items

def load_checkpoint():
    """Load progress from checkpoint file"""
    if os.path.exists(CHECKPOINT_FILE):
        try:
            with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
                return json.load(f)
        except:
            return {"processed_tmdb_ids": [], "episode_dicts": []}
    return {"processed_tmdb_ids": [], "episode_dicts": []}

def save_checkpoint(processed_tmdb_ids, episode_dicts):
    """Save progress to checkpoint file"""
    checkpoint = {
        "processed_tmdb_ids": processed_tmdb_ids,
        "episode_dicts": episode_dicts,
        "timestamp": datetime.now().isoformat(),
        "total_processed": len(processed_tmdb_ids),
        "total_episodes": len(episode_dicts)
    }
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump(checkpoint, f, indent=2, ensure_ascii=False)

def save_final_output(episode_dicts):
    """Save final results to output file"""
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(episode_dicts, f, indent=2, ensure_ascii=False)

# Load checkpoint
checkpoint = load_checkpoint()
existing_episode_dicts = checkpoint.get("episode_dicts", [])
processed_tmdb_ids = set(checkpoint.get("processed_tmdb_ids", []))

print("="*80)
print("EXTRACTING EPISODE INFORMATION FROM WIKIPEDIA FOR TV SHOWS")
print("="*80)
print(f"Total TV shows fetched from TMDB: {len(tv_shows)}")
print(f"Already processed (from checkpoint): {len(processed_tmdb_ids)}")
print(f"Total episodes extracted so far: {len(existing_episode_dicts)}")

# Filter TV shows to only process those NOT in checkpoint
tv_shows_to_process = [show for show in tv_shows if show.get("tmdb_id") not in processed_tmdb_ids]

print(f"New TV shows to process: {len(tv_shows_to_process)}")
print(f"Request delay: {REQUEST_DELAY} seconds (to avoid blocking)")
print("="*80)

if not tv_shows_to_process:
    print("\n✓ All TV shows from TMDB are already in the checkpoint!")
    print("  No new TV shows to process.")
else:
    print("\n⚠️  IMPORTANT: This will take a while!")
    print(f"   Estimated time: ~{len(tv_shows_to_process) * REQUEST_DELAY / 60:.1f} minutes")
    print("   Progress is saved automatically - you can stop and resume anytime")
    print("   Extracting EPISODE information from Wikipedia\n")

# Start with existing episode dictionaries
episode_dicts = existing_episode_dicts.copy()

if tv_shows_to_process:
    for idx, tv_show in enumerate(tqdm(tv_shows_to_process, desc="Processing"), 1):
        try:
            # Search for Wikipedia page
            page_title = search_wikipedia(
                tv_show["title"],
                tv_show.get("year"),
                tv_show.get("media_type")
            )
            
            if page_title:
                # Get episodes from Wikipedia
                episodes_data = get_wikipedia_episodes(page_title, debug=False)
                
                if episodes_data:
                    # Handle both dict and Wikicode object
                    if hasattr(episodes_data, 'strip_code'):
                        # It's a Wikicode object, convert to dict
                        raw_wikicode = str(episodes_data)
                        content = episodes_data.strip_code().strip()
                        episodes_data = {
                            "section": "Episodes",
                            "content": content[:50000],
                            "raw_wikicode": raw_wikicode[:50000]
                        }
                    
                    # Get wikicode object to parse templates
                    if isinstance(episodes_data, dict):
                        raw_wikicode_str = episodes_data.get("raw_wikicode", "")
                        if raw_wikicode_str:
                            episodes_wikicode = mwparserfromhell.parse(raw_wikicode_str)
                        else:
                            episodes_wikicode = None
                    else:
                        episodes_wikicode = episodes_data
                    
                    # Parse {{Episode list}} templates using mwparserfromhell
                    parsed_episodes = []
                    
                    if episodes_wikicode:
                        # Use mwparserfromhell to find all Episode list templates
                        episode_templates = episodes_wikicode.filter_templates(matches=lambda t: t.name.strip().lower() == 'episode list')
                        
                        for template in episode_templates:
                            # Extract parameters from template
                            ep_num = None
                            title = ""
                            summary = ""
                            
                            # Get EpisodeNumber parameter
                            if template.has('EpisodeNumber'):
                                ep_num_str = str(template.get('EpisodeNumber').value).strip()
                                try:
                                    ep_num = int(ep_num_str)
                                except ValueError:
                                    pass
                            
                            # Get Title parameter
                            if template.has('Title'):
                                title = str(template.get('Title').value).strip()
                            
                            # Get ShortSummary parameter
                            if template.has('ShortSummary'):
                                summary = str(template.get('ShortSummary').value).strip()
                            
                            # Clean up text
                            def clean_wiki_text(text):
                                if not text:
                                    return ""
                                try:
                                    parsed = mwparserfromhell.parse(text)
                                    cleaned = parsed.strip_code().strip()
                                    return cleaned
                                except:
                                    text = re.sub(r'\[\[([^\|]+?)(?:\|([^\]]+?))?\]\]', lambda m: m.group(2) if m.group(2) else m.group(1), text)
                                    text = re.sub(r'\{\{[^}]+\}\}', '', text)
                                    text = re.sub(r'<ref[^>]*/>', '', text)
                                    text = re.sub(r'<ref[^>]*>.*?</ref>', '', text, flags=re.DOTALL)
                                    text = re.sub(r'<!--.*?-->', '', text, flags=re.DOTALL)
                                    return text.strip()
                            
                            title = clean_wiki_text(title)
                            summary = clean_wiki_text(summary)
                            
                            if ep_num:
                                parsed_episodes.append({
                                    "episode_number": ep_num,
                                    "title": title,
                                    "summary": summary
                                })
                    
                    # Filter valid episodes and create dictionaries
                    valid_episodes = [ep for ep in parsed_episodes if ep.get('episode_number') and (ep.get('title') or ep.get('summary'))]
                    
                    if valid_episodes:
                        # Create separate dictionaries for each episode
                        tv_show_title = tv_show["title"]
                        for ep in valid_episodes:
                            ep_num = ep.get('episode_number', '')
                            summary = ep.get('summary', '')
                            
                            # Create key: title_episode1, title_episode2, etc.
                            episode_key = f"{tv_show_title}_episode{ep_num}"
                            
                            # Use summary as the plot value
                            plot_value = summary if summary else ""
                            
                            # Create dictionary for this episode with TMDB information
                            episode_dict = {
                                episode_key: plot_value,
                                "tmdb_id": tv_show.get("tmdb_id"),
                                "title": tv_show.get("title"),
                                "year": tv_show.get("year"),
                                "first_air_date": tv_show.get("first_air_date"),
                                "media_type": tv_show.get("media_type"),
                                "episode_number": ep_num
                            }
                            episode_dicts.append(episode_dict)
            
            # Mark as processed
            processed_tmdb_ids.add(tv_show.get("tmdb_id"))
            
            # Save checkpoint periodically
            if idx % BATCH_SAVE_INTERVAL == 0:
                save_checkpoint(list(processed_tmdb_ids), episode_dicts)
                print(f"\n  ✓ Checkpoint saved: {idx}/{len(tv_shows_to_process)} TV shows processed, {len(episode_dicts)} total episodes")
            
        except requests.exceptions.HTTPError as e:
            if e.response.status_code == 403:
                print(f"\n⚠️  403 Forbidden detected! Pausing for 60 seconds...")
                print(f"   Processed {idx}/{len(tv_shows_to_process)} so far")
                save_checkpoint(list(processed_tmdb_ids), episode_dicts)
                time.sleep(60)
                continue
            else:
                # Other HTTP errors - skip this item
                processed_tmdb_ids.add(tv_show.get("tmdb_id"))
        except Exception as e:
            # Other errors - skip this item but record it
            processed_tmdb_ids.add(tv_show.get("tmdb_id"))
        
        # Rate limiting delay
        time.sleep(REQUEST_DELAY)

# Final save
save_checkpoint(list(processed_tmdb_ids), episode_dicts)
save_final_output(episode_dicts)

# Print summary
print("\n" + "="*80)
print("EXTRACTION COMPLETE")
print("="*80)
print(f"Total TV shows processed: {len(processed_tmdb_ids)}")
print(f"Total episodes extracted: {len(episode_dicts)}")
print(f"\nResults saved to: {OUTPUT_FILE}")
print(f"Checkpoint saved to: {CHECKPOINT_FILE}")
print("="*80)

# Show some statistics
if episode_dicts:
    print(f"\n💡 Format: Each episode is stored as a separate dictionary with TMDB info:")
    print(f"   {{'TV_Show_Title_episode1': 'plot text', 'tmdb_id': ..., 'title': ..., 'year': ..., etc.}}")
    print(f"   {{'TV_Show_Title_episode2': 'plot text', 'tmdb_id': ..., 'title': ..., 'year': ..., etc.}}")
    print(f"   etc.")
    
    # Show example
    example = episode_dicts[0] if episode_dicts else None
    if example:
        # Find the episode key (the one that contains '_episode')
        episode_key = None
        for key in example.keys():
            if '_episode' in key:
                episode_key = key
                break
        
        if episode_key:
            print(f"\nExample episode dictionary:")
            print(f"  {episode_key}: {example[episode_key][:100]}...")
            print(f"  tmdb_id: {example.get('tmdb_id')}")
            print(f"  title: {example.get('title')}")
            print(f"  year: {example.get('year')}")
            print(f"  episode_number: {example.get('episode_number')}")

EXTRACTING EPISODE INFORMATION FROM WIKIPEDIA FOR TV SHOWS
Total TV shows fetched from TMDB: 2000
Already processed (from checkpoint): 0
Total episodes extracted so far: 0
New TV shows to process: 2000
Request delay: 1.5 seconds (to avoid blocking)

⚠️  IMPORTANT: This will take a while!
   Estimated time: ~50.0 minutes
   Progress is saved automatically - you can stop and resume anytime
   Extracting EPISODE information from Wikipedia



Processing:   0%|          | 0/2000 [00:00<?, ?it/s]

Processing:   0%|          | 4/2000 [00:12<1:44:07,  3.13s/it]


  ✓ Checkpoint saved: 5/2000 TV shows processed, 23 total episodes


Processing:   0%|          | 9/2000 [00:28<1:45:34,  3.18s/it]

...


  ✓ Checkpoint saved: 1995/2000 TV shows processed, 12251 total episodes


Processing: 100%|█████████▉| 1999/2000 [1:45:21<00:03,  3.16s/it]


  ✓ Checkpoint saved: 2000/2000 TV shows processed, 12276 total episodes


Processing: 100%|██████████| 2000/2000 [1:45:25<00:00,  3.16s/it]



EXTRACTION COMPLETE
Total TV shows processed: 2000
Total episodes extracted: 12276

Results saved to: tmdb_wikipedia_tv_episodes.json
Checkpoint saved to: tv_episode_extraction_checkpoint.json

💡 Format: Each episode is stored as a separate dictionary with TMDB info:
   {'TV_Show_Title_episode1': 'plot text', 'tmdb_id': ..., 'title': ..., 'year': ..., etc.}
   {'TV_Show_Title_episode2': 'plot text', 'tmdb_id': ..., 'title': ..., 'year': ..., etc.}
   etc.

Example episode dictionary:
  The Four Seasons_episode1: In the spring, married friends Nick and Anne, Kate and Jack, and Danny and Claude spend a weekend at...
  tmdb_id: 243316
  title: The Four Seasons
  year: 2025
  episode_number: 1


### Clean fetched data
- remove episodes without plots

In [ ]:
# pre-processings
with open("../raw_data/tv_raw/tv_episode_extraction_checkpoint.json", "r") as f:
    data = json.load(f)


In [ ]:

# Filter out episodes with empty plots
filtered = [i for i in data["episode_dicts"] if list(i.values())[0] != ""]

# Save to JSON
with open("../raw_data/tv_raw/filtered_episode_dicts.json", "w", encoding="utf-8") as f:
    json.dump(filtered, f, indent=2, ensure_ascii=False)

#### Optional: use wikipedia fetch some more tv shows

In [ ]:
WIKI_API = "https://en.wikipedia.org/w/api.php"
HEADERS = {
    "User-Agent": "RAGTVShowBot/1.0 (contact: your_email@domain.com)"
}

def get_tv_shows_by_category(category, max_pages=500):
    """Fetch all pages in a given Wikipedia category (TV shows)."""
    shows = []
    cmcontinue = None

    while True:
        params = {
            "action": "query",
            "list": "categorymembers",
            "cmtitle": f"Category:{category}",
            "cmtype": "page",        # only real pages, skip subcategories
            "cmlimit": 500,          # max per request
            "format": "json"
        }
        if cmcontinue:
            params["cmcontinue"] = cmcontinue

        response = requests.get(WIKI_API, params=params, headers=HEADERS)
        response.raise_for_status()
        data = response.json()

        pages = data.get("query", {}).get("categorymembers", [])
        shows.extend(pages)

        # Pagination
        cmcontinue = data.get("continue", {}).get("cmcontinue")
        if not cmcontinue or len(shows) >= max_pages:
            break

    return shows[:max_pages]  # trim to max_pages if needed

In [ ]:
# get show from 2024,2025,2026

years = [2024, 2025, 2026]
all_tv_shows = []

for year in years:
    category_name = f"{year} American television series debuts"
    shows = get_tv_shows_by_category(category_name)
    print(f"{year}: {len(shows)} shows found")
    all_tv_shows.extend(shows)

In [11]:
# Check all_tv_shows against merged_chunks.json
# Parse episodes for shows not in merged_chunks, skip if no episodes found

print("Checking all_tv_shows against merged_chunks.json...")
print("=" * 80)

# Load merged_chunks.json to check for existing TV shows
try:
    with open("../chunk_data/merged_chunks.json", "r", encoding="utf-8") as f:
        merged_chunks = json.load(f)
    
    # Extract existing TV show titles (normalized)
    existing_tv_shows = set()
    for chunk in merged_chunks:
        chunk_id = chunk.get("id", "")
        if chunk_id.startswith("t_"):  # TV show chunk
            metadata = chunk.get("metadata", {})
            title = metadata.get("title", "")
            if title:
                # Normalize title for comparison (lowercase, remove extra spaces)
                normalized_title = title.lower().strip()
                existing_tv_shows.add(normalized_title)
    
    print(f"✓ Loaded {len(existing_tv_shows)} existing TV shows from merged_chunks")
except FileNotFoundError:
    print("⚠️  merged_chunks.json not found. All shows will be considered new.")
    existing_tv_shows = set()

print("=" * 80)

# Check which shows from all_tv_shows are not in merged_chunks
print(f"\nTotal shows in all_tv_shows: {len(all_tv_shows)}")

shows_to_process = []
for show in all_tv_shows:
    show_title = show.get("title", "")
    if not show_title:
        continue
    
    # Normalize title for comparison
    normalized_title = show_title.lower().strip()
    
    # Remove year markers and (TV series) markers for comparison
    normalized_title = re.sub(r'\s*\(\d{4}.*?\)', '', normalized_title)
    normalized_title = re.sub(r'\s*\(TV series\)', '', normalized_title, flags=re.IGNORECASE)
    normalized_title = normalized_title.strip()
    
    # Check if any existing show matches (fuzzy match)
    found = False
    for existing_title in existing_tv_shows:
        existing_normalized = re.sub(r'\s*\(\d{4}.*?\)', '', existing_title)
        existing_normalized = re.sub(r'\s*\(TV series\)', '', existing_normalized, flags=re.IGNORECASE)
        existing_normalized = existing_normalized.strip()
        
        # Check if titles match (allowing for minor variations)
        if normalized_title == existing_normalized:
            found = True
            break
    
    if not found:
        shows_to_process.append(show)

print(f"Shows NOT in merged_chunks: {len(shows_to_process)}")
print(f"Shows already in merged_chunks: {len(all_tv_shows) - len(shows_to_process)}")
print("=" * 80)

# Output file for new episodes
OUTPUT_FILE = "../raw_data/tv_raw/wikipedia_2024_2026_tv_episodes.json"
REQUEST_DELAY = 1.5
BATCH_SAVE_INTERVAL = 5

# Load existing output file if it exists
try:
    with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
        existing_episode_dicts = json.load(f)
    print(f"\n✓ Loaded {len(existing_episode_dicts)} existing episodes from output file")
except FileNotFoundError:
    existing_episode_dicts = []
    print("\nℹ️  No existing output file found, will create new one")

# Process shows and fetch episodes
print(f"\n{'='*80}")
print("FETCHING EPISODES FOR NEW TV SHOWS")
print("="*80)
print(f"Total shows to process: {len(shows_to_process)}")
print(f"Estimated time: ~{len(shows_to_process) * REQUEST_DELAY / 60:.1f} minutes")
print("="*80)

episode_dicts = existing_episode_dicts.copy()
new_episodes_count = 0
skipped_no_episodes = 0
skipped_error = 0

for idx, tv_show_page in enumerate(tqdm(shows_to_process, desc="Processing shows"), 1):
    try:
        show_title = tv_show_page.get("title", "")
        if not show_title:
            continue
        
        # Skip "List of..." pages
        if "list of" in show_title.lower():
            continue
        
        page_title = show_title  # Use the title directly from Wikipedia
        
        # Get episodes from Wikipedia
        episodes_data = get_wikipedia_episodes(page_title, debug=False)
        
        if not episodes_data:
            skipped_no_episodes += 1
            continue
        
        # Handle both dict and Wikicode object
        if hasattr(episodes_data, 'strip_code'):
            raw_wikicode = str(episodes_data)
            content = episodes_data.strip_code().strip()
            episodes_data = {
                "section": "Episodes",
                "content": content[:50000],
                "raw_wikicode": raw_wikicode[:50000]
            }
        
        # Get wikicode object to parse templates
        if isinstance(episodes_data, dict):
            raw_wikicode_str = episodes_data.get("raw_wikicode", "")
            if raw_wikicode_str:
                episodes_wikicode = mwparserfromhell.parse(raw_wikicode_str)
            else:
                skipped_no_episodes += 1
                continue
        else:
            episodes_wikicode = episodes_data
        
        # Parse {{Episode list}} templates
        parsed_episodes = []
        
        if episodes_wikicode:
            episode_templates = episodes_wikicode.filter_templates(
                matches=lambda t: t.name.strip().lower() == 'episode list'
            )
            
            for template in episode_templates:
                ep_num = None
                title = ""
                summary = ""
                
                if template.has('EpisodeNumber'):
                    ep_num_str = str(template.get('EpisodeNumber').value).strip()
                    try:
                        ep_num = int(ep_num_str)
                    except ValueError:
                        pass
                
                if template.has('Title'):
                    title = str(template.get('Title').value).strip()
                
                if template.has('ShortSummary'):
                    summary = str(template.get('ShortSummary').value).strip()
                
                def clean_wiki_text(text):
                    if not text:
                        return ""
                    try:
                        parsed = mwparserfromhell.parse(text)
                        cleaned = parsed.strip_code().strip()
                        return cleaned
                    except:
                        text = re.sub(r'\[\[([^\|]+?)(?:\|([^\]]+?))?\]\]', 
                                    lambda m: m.group(2) if m.group(2) else m.group(1), text)
                        text = re.sub(r'\{\{[^}]+}\}', '', text)
                        text = re.sub(r'<ref[^>]*/>', '', text)
                        text = re.sub(r'<ref[^>]*>.*?</ref>', '', text, flags=re.DOTALL)
                        text = re.sub(r'<!--.*?-->', '', text, flags=re.DOTALL)
                        return text.strip()
                
                title = clean_wiki_text(title)
                summary = clean_wiki_text(summary)
                
                if ep_num:
                    parsed_episodes.append({
                        "episode_number": ep_num,
                        "title": title,
                        "summary": summary
                    })
        
        # Filter valid episodes
        valid_episodes = [ep for ep in parsed_episodes 
                        if ep.get('episode_number') and (ep.get('title') or ep.get('summary'))]
        
        if not valid_episodes:
            skipped_no_episodes += 1
            continue
        
        # Extract base title
        base_title = re.sub(r'\s*\(\d{4}.*?\)', '', show_title).strip()
        base_title = re.sub(r'\s*\(TV series\)', '', base_title, flags=re.IGNORECASE).strip()
        
        # Create episode dictionaries
        show_new_count = 0
        for ep in valid_episodes:
            ep_num = ep.get('episode_number', '')
            summary = ep.get('summary', '')
            
            # Create episode key
            episode_key = f"{base_title}_episode{ep_num}"
            
            # Create dictionary for this episode
            episode_dict = {
                episode_key: summary if summary else "",
                "tmdb_id": None,  # No TMDB ID for Wikipedia-only shows
                "title": base_title,
                "year": "",  # Could extract from Wikipedia if needed
                "first_air_date": "",
                "media_type": "tv",
                "episode_number": ep_num,
                "wikipedia_title": page_title
            }
            episode_dicts.append(episode_dict)
            new_episodes_count += 1
            show_new_count += 1
        
        if show_new_count > 0:
            print(f"\n  ✓ {show_title}: Added {show_new_count} episodes")
    
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 403:
            print(f"\n⚠️  403 Forbidden detected! Pausing for 60 seconds...")
            print(f"   Processed {idx}/{len(shows_to_process)} so far")
            time.sleep(60)
            continue
        else:
            skipped_error += 1
            print(f"\n⚠️  Error for '{show_title}': {e}")
    except Exception as e:
        skipped_error += 1
        print(f"\n⚠️  Error for '{show_title}': {e}")
    
    # Periodic save
    if idx % BATCH_SAVE_INTERVAL == 0:
        with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
            json.dump(episode_dicts, f, indent=2, ensure_ascii=False)
        print(f"\n  💾 Progress saved: {idx}/{len(shows_to_process)} shows processed")
        print(f"     New episodes: {new_episodes_count}, Skipped (no episodes): {skipped_no_episodes}")
    
    # Rate limiting delay
    time.sleep(REQUEST_DELAY)

# Final save
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(episode_dicts, f, indent=2, ensure_ascii=False)

print(f"\n{'='*80}")
print("FETCHING COMPLETE - SUMMARY")
print("="*80)
print(f"Total shows processed: {len(shows_to_process)}")
print(f"  ✓ New episodes added: {new_episodes_count}")
print(f"  ✗ Skipped (no episodes): {skipped_no_episodes}")
print(f"  ⚠️  Errors: {skipped_error}")
print(f"\n✓ Total episodes in output file: {len(episode_dicts)}")
print(f"✓ Results saved to: {OUTPUT_FILE}")
print("="*80)

Checking all_tv_shows against merged_chunks.json...
✓ Loaded 461 existing TV shows from merged_chunks

Total shows in all_tv_shows: 442
Shows NOT in merged_chunks: 300
Shows already in merged_chunks: 142

ℹ️  No existing output file found, will create new one

FETCHING EPISODES FOR NEW TV SHOWS
Total shows to process: 300
Estimated time: ~7.5 minutes


Processing shows:   0%|          | 0/300 [00:00<?, ?it/s]


  ✓ The 1% Club (American game show): Added 12 episodes


Processing shows:   1%|▏         | 4/300 [00:04<04:34,  1.08it/s]


  ✓ Star Wars: The Acolyte: Added 8 episodes

  💾 Progress saved: 5/300 shows processed
     New episodes: 20, Skipped (no episodes): 2


...


  ✓ Memory of a Killer (TV series): Added 4 episodes


Processing shows: 100%|█████████▉| 299/300 [08:57<00:02,  2.24s/it]


  ✓ Wonder Man (miniseries): Added 8 episodes

  💾 Progress saved: 300/300 shows processed
     New episodes: 2375, Skipped (no episodes): 89


Processing shows: 100%|██████████| 300/300 [08:59<00:00,  1.80s/it]


FETCHING COMPLETE - SUMMARY
Total shows processed: 300
  ✓ New episodes added: 2375
  ✗ Skipped (no episodes): 89
  ⚠️  Errors: 0

✓ Total episodes in output file: 2375
✓ Results saved to: ../raw_data/tv_raw/wikipedia_2024_2026_tv_episodes.json


In [ ]:
# processings
with open("../raw_data/tv_raw/wikipedia_2024_2026_tv_episodes.json", "r") as f:
    data = json.load(f)
# Filter out episodes with empty plots
filtered = [i for i in data if list(i.values())[0] != ""]

# Save to JSON
with open("../raw_data/tv_raw/wikipedia_2024_2026_tv_episodes.json", "w", encoding="utf-8") as f:
    json.dump(filtered, f, indent=2, ensure_ascii=False)